# 🁫 Domino RL — Entrenamiento en Colab (CPU)

Entrena al agente de dominó cubano doble-9 con self-play. **Usa runtime de CPU**
(Entorno de ejecución → Cambiar tipo de entorno → Acelerador: *None*): la red es
diminuta (143→128→128→111) y el cuello de botella es simular partidas, no la GPU.
No gastes tu cuota de GPU aquí.

**Camino progresivo:** 1v1 (2 jugadores) → 4 individual → parejas 2v2.
Los checkpoints se guardan en tu Google Drive: si Colab se desconecta,
re-ejecutas con `--resume` y sigue donde quedó.

## 0. Clonar el repo (privado — te pedirá tu token de GitHub)

In [ ]:
import getpass, os
token = getpass.getpass('GitHub token (no se muestra): ')
os.system(f'git clone https://{token}@github.com/SRogDev/domino-rl.git')
del token

## 1. Instalar dependencias

In [ ]:
!pip install -q gymnasium numpy stable-baselines3 sb3-contrib
!cd domino-rl && python -m pytest tests/ -q 2>&1 | tail -1

## 2. Montar Drive (checkpoints sobreviven desconexiones)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/domino-rl'
import os; os.makedirs(OUT, exist_ok=True)
print('checkpoints en:', OUT)

## 3. Paso 1 — 1v1 con 2 jugadores (~30-45 min, 1M pasos)

El problema más simple con información oculta real. Solo valida que el
pipeline aprende: si no le gana al azar aquí, algo está roto (no son los
hiperparámetros).

In [ ]:
!cd domino-rl && PYTHONPATH=src python -m domino_rl.train \
  --n-players 2 --mode individual --reward-mode sparse \
  --timesteps 1000000 --n-envs 8 --seed 0 \
  --out $OUT/1v1 --resume --ckpt-every 250000

### Evaluar el 1v1: ¿aprendió algo?

In [ ]:
!cd domino-rl && PYTHONPATH=src python -m domino_rl.evaluate --rounds 500 \
  --a ppo:$OUT/1v1/final.zip --b random --mode individual --n-players 2
!cd domino-rl && PYTHONPATH=src python -m domino_rl.evaluate --rounds 500 \
  --a ppo:$OUT/1v1/final.zip --b greedy --mode individual --n-players 2

✅ **Listo para el paso 2 si:** le gana a `random` >60% y a `greedy` >50%.

## 4. Paso 2 — 4 jugadores individual (~1-2h, 2M pasos)

El mismo cerebro, ahora con 4 manos ocultas. Inicializa desde el 1v1
copiando el checkpoint:

In [ ]:
!mkdir -p $OUT/4p && cp $OUT/1v1/final.zip $OUT/4p/final.zip  # warm start
!cd domino-rl && PYTHONPATH=src python -m domino_rl.train \
  --mode individual --reward-mode sparse \
  --timesteps 2000000 --n-envs 8 --seed 1 \
  --out $OUT/4p --resume --ckpt-every 250000

In [ ]:
!cd domino-rl && PYTHONPATH=src python -m domino_rl.evaluate --rounds 500 \
  --a ppo:$OUT/4p/final.zip --b greedy --mode individual

## 5. Paso 3 — Parejas 2v2, el objetivo real (~3-6h, 3-5M pasos)

Aquí debe emerger la coordinación ciega con la pareja. Warm start desde el 4p.
Puedes dividirlo en varias sesiones: `--resume` continúa donde quedó.

In [ ]:
!mkdir -p $OUT/teams && cp $OUT/4p/final.zip $OUT/teams/final.zip  # warm start
!cd domino-rl && PYTHONPATH=src python -m domino_rl.train \
  --mode teams --reward-mode sparse \
  --timesteps 3000000 --n-envs 8 --seed 2 \
  --out $OUT/teams --resume --ckpt-every 250000

In [ ]:
!cd domino-rl && PYTHONPATH=src python -m domino_rl.evaluate --rounds 1000 \
  --a ppo:$OUT/teams/final.zip --b greedy --mode teams

## 6. Llevar la campeona a la mesa digital

Descarga `$OUT/teams/final.zip`, renómbralo a `ppo_smoke.zip` y ponlo en
`demo/models/` del repo: la demo web la usará como rival y la opción
"PPO estudiante" aparecerá sola en el menú.